# Exploratory Data Analysis

## Imports & Initial Setups

In [42]:
import numpy as np
import pandas as pd
from pathlib import Path

In [67]:
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option('display.max_columns', None)

## Loading Dataset

Here we're loading just one small chunk of the whole CSV, we can later apply our learnings here one complete dataset to confirm our findings across the whole dataset.

In [44]:
df = pd.read_csv(Path('../dataset/extracted/labeled/Friday-02-03-2018.csv'), chunksize=1_000_000, low_memory=False).get_chunk()

In [45]:
header = df.columns

header_rows = (df.astype(str) == header).all(axis=1)

print(list(header_rows).count(True))

from src.core.config import config_loader

df = df.loc[~header_rows].copy()

schema_cfg = config_loader('../config/preprocessing/validation_schema.yaml')
non_numeric_cols = schema_cfg['non_numeric_col']

feature_cols = df.columns.drop(non_numeric_cols)

df[feature_cols] = df[feature_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

0


We found that our dataset had zero header duplicates.

## Analysis

In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Columns: 117 entries, Flow ID to Label
dtypes: float64(87), int64(25), str(5)
memory usage: 981.8 MB


In [47]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)


In [48]:
df.describe()

       Source Port  Destination Port  Protocol  Flow Duration  \
count    999997.00         999997.00 999997.00      999997.00   
mean      51154.09           1435.69     10.06    18934899.34   
std       12880.08           3549.47      5.31    37590408.46   
min          15.00              0.00      6.00           0.00   
25%       49799.00             53.00      6.00        1111.00   
50%       52404.00            443.00      6.00      217964.00   
75%       58801.00           3389.00     17.00     4583542.00   
max       65535.00          65535.00     17.00   119999997.00   

       Total Fwd Packets  Total Bwd Packets  Total Length of Fwd Packets  \
count          999997.00          999997.00                    999997.00   
mean                7.26               9.65                       551.65   
std                88.40             325.02                      2043.74   
min                 0.00               0.00                         0.00   
25%                 1.00          

In [49]:
negative_counts = (df.select_dtypes(include="number") < 0).sum()
negative_counts[negative_counts > 0]

Flow Bytes/s                  45302
Flow Packets/s                45302
TTL Fwd Mean                   3762
TTL Fwd Std                    3762
TTL Fwd Min                    3762
TTL Fwd Max                    3762
TTL Bwd Mean                  53770
TTL Bwd Std                   53770
TTL Bwd Min                   53770
TTL Bwd Max                   53770
TCP Window Fwd Mean          373193
TCP Window Fwd Std           373193
TCP Window Fwd Min           373193
TCP Window Fwd Max           373193
TCP Window Bwd Mean          410602
TCP Window Bwd Std           410602
TCP Window Bwd Min           410602
TCP Window Bwd Max           410602
Fragmentation Offset Mean    999997
Fragmentation Offset Max     999997
Payload Mean                     25
Payload Min                   34313
Payload P25                      35
dtype: int64

In [62]:
for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        print(col, df[df[col] < 0][col].unique())

Source Port []
Destination Port []
Protocol []
Flow Duration []
Total Fwd Packets []
Total Bwd Packets []
Total Length of Fwd Packets []
Total Length of Bwd Packets []
Fwd Packet Length Max []
Fwd Packet Length Min []
Fwd Packet Length Mean []
Fwd Packet Length Std []
Bwd Packet Length Max []
Bwd Packet Length Min []
Bwd Packet Length Mean []
Bwd Packet Length Std []
Flow Bytes/s [-1.]
Flow Packets/s [-1.]
Flow IAT Mean []
Flow IAT Std []
Flow IAT Max []
Flow IAT Min []
Fwd IAT Total []
Fwd IAT Mean []
Fwd IAT Std []
Fwd IAT Max []
Fwd IAT Min []
Bwd IAT Total []
Bwd IAT Mean []
Bwd IAT Std []
Bwd IAT Max []
Bwd IAT Min []
Fwd PSH Flags []
Bwd PSH Flags []
Fwd URG Flags []
Bwd URG Flags []
Fwd Header Length []
Bwd Header Length []
Fwd Packets/s []
Bwd Packets/s []
Packet Length Min []
Packet Length Max []
Packet Length Mean []
Packet Length Std []
Packet Length Variance []
FIN Flag Count []
SYN Flag Count []
RST Flag Count []
PSH Flag Count []
ACK Flag Count []
URG Flag Count []
CWE Fl

From the above we can see many columns having negative value. Some of these such as TTL statistics were given the sentinel value of -1 by us.

However, the negative values in columns `Payload P25`, `Payload Min`, `Payload Mean`, `Flow Bytes/s` and `Flow Packets/s` do not make sense, these are likely invalid values.

In [54]:
for col in df.columns:
    if df[col].dtype in ['int64', 'float64'] and df[col].var() == 0:
        print(col, end=" ")
        print(df[col].unique())

Fwd URG Flags [0]
Bwd URG Flags [0]
URG Flag Count [0]
dst_port_sequentiality [0.]
Fragmentation Count [0.]
Fragmentation Offset Mean [-1.]
Fragmentation Offset Max [-1.]


In the selected chunk, the following  columns seem to have zero variance:

- Fwd URG Flags
- Bwd URG Flags
- dst_port_sequentiality
- Fragmentation Count
- Fragmentation Offset Mean
- Fragmentation Offset Max

However, we believe that may not be true in other cases, however as the system is to remain largely configurable we intend to drop these columns should these be constant throughout the dataset.

In [70]:
corr = df.corr(numeric_only=True)
out_data_path = Path('../out/data', "Friday-02-03-2018.csv")
out_data_path.mkdir(exist_ok=True, parents=True)
corr.to_csv(Path(out_data_path, 'corr.csv'))

corr

,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Bwd Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Packet Length Min,Packet Length Max,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,dst_port_unique_cnt,dst_port_scan_rate,dst_port_sequentiality,dst_port_entropy,src_dst_pair_flow_rate,TTL Fwd Mean,TTL Fwd Std,TTL Fwd Min,TTL Fwd Max,TTL Bwd Mean,TTL Bwd Std,TTL Bwd Min,TTL Bwd Max,TCP Window Fwd Mean,TCP Window Fwd Std,TCP Window Fwd Min,TCP Window Fwd Max,TCP Window Bwd Mean,TCP Window Bwd Std,TCP Window Bwd Min,TCP Window Bwd Max,Fragmentation Count,Fragmentation Offset Mean,Fragmentation Offset Max,Payload Mean,Payload Std,Payload Min,Payload Max,Payload Median,Payload P25,Payload P75,Retransmission Count
Source Port,1.00,-0.17,0.26,-0.32,-0.02,-0.00,-0.06,-0.00,-0.17,-0.00,-0.11,-0.12,-0.08,0.27,0.04,-0.07,0.08,0.01,-0.25,-0.38,-0.42,-0.07,-0.06,-0.12,-0.04,-0.07,-0.12,-0.31,-0.24,-0.39,-0.41,-0.06,-0.12,-0.01,NaN,NaN,-0.02,-0.00,-0.00,0.06,-0.00,-0.11,-0.02,-0.05,-0.04,-0.20,-0.04,-0.15,-0.03,-0.01,NaN,-0.13,-0.16,0.16,-0.02,-0.11,0.04,-0.02,-0.01,-0.03,-0.01,-0.00,-0.00,-0.01,-0.08,-0.20,-0.01,-0.00,-0.17,-0.17,-0.04,-0.24,-0.28,-0.01,-0.25,-0.28,-0.43,-0.02,-0.42,-0.43,0.20,0.00,NaN,0.22,-0.04,0.10,-0.01,0.10,0.10,0.34,0.02,0.34,0.34,-0.05,-0.02,-0.12,-0.09,-0.12,-0.37,0.13,-0.17,NaN,NaN,NaN,-0.02,-0.05,0.04,-0.11,0.04,0.04,0.00,-0.01
Destination Port,-0.17,1.00,-0.26,0.02,0.00,-0.00,0.07,-0.00,0.24,-0.17,0.19,0.24,0.16,-0.25,-0.02,0.15,-0.08,0.02,0.02,0.07,0.09,0.00,-0.08,-0.04,-0.07,-0.06,0.02,0.01,0.02,0.08,0.09,-0.01,0.11,0.02,NaN,NaN,0.00,-0.00,0.04,-0.07,-0.17,0.17,0.02,0.11,0.05,-0.08,0.04,0.21,0.04,-0.00,NaN,0.17,0.20,-0.20,0.02,0.19,-0.02,0.00,0.02,0.06,0.03,-0.00,-0.00,-0.03,0.02,0.09,0.00,-0.00,0.14,0.28,0.03,0.25,0.09,-0.01,0.08,0.10,0.10,-0.04,0.09,0.10,-0.39,0.07,NaN,-0.40,-0.03,-0.00,0.01,-0.00,-0.00,-0.31,-0.06,-0.31,-0.31,0.04,0.07,0.04,0.09,0.32,0.03,0.27,0.27,NaN,NaN,NaN,0.02,0.11,-0.17,0.17,-0.08,-0.14,-0.03,-0.03
Protocol,0.26,-0.26,1.00,-0.38,-0.05,-0.02,-0.18,-0.01,-0.58,0.74,-0.30,-0.64,-0.59,0.82,-0.23,-0.62,0.26,0.08,-0.20,-0.32,-0.32,0.02,-0.29,-0.15,-0.25,-0.24,0.03,-0.37,-0.23,-0.32,-0.32,-0.05,-0.36,-0.07,NaN,NaN,-0.06,-0.02,0.04,0.15,0.73,-0.61,-0.25,-0.55,-0.45,-0.42,-0.14,-0.61,-0.14,-0.03,NaN,-0.50,-0.50,0.10,-0.25,-0.30,-0.23,-0.06,-0.01,-0.02,-0.00,-0.01,-0.01,-0.07,-0.09,-0.20,-0.02,-0.01,-0.53,-0.64,-0.12,-0.90,-0.18,-0.07,-0.19,-0.17,-0.31,-0.11,-0.31,-0.29,0.52,-0.15,NaN,0.56,-0.20,0.21,-0.02,0.21,0.21,0.73,-0.04,0.73,0.72,-0.27,-0.36,-0.18,-0.45,-0.51,-0.42,-0.39,-0.65,NaN,NaN,NaN,-0.25,-0.55,0.73,-0.61,0.15,0.52,-0.12,-0.11
Flow Duration,-0.32,0.02,-0.38,1.00,0.08,0.04,0.20,0.02,0.41,-0.27,0.13,0.34,0.41,-0.30,0.26,0.36,-0.15,-0.05,0.55,0.79,0.80,0.04,0.80,0.46,0.56,0.59,0.08,0.99,0.57,0.77,0.79,0.13,0.31,0.07,NaN,NaN,0.07,0.04,-0.03,-0.10,-0.27,0.44,0.26,0.39,0.3

## Perfect Positive Correlation (r = +1.000)

| Feature 1              | Feature 2            | Correlation |
|------------------------|----------------------|------------:|
| Fwd Packet Length Mean | Avg Fwd Segment Size |   +1.000000 |
| Bwd Packet Length Mean | Avg Bwd Segment Size |   +1.000000 |
| Fwd Header Length      | Fwd Header Length.1  |   +1.000000 |
| Packet Length Max      | Payload Max          |   +1.000000 |
| Packet Length Mean     | Average Packet Size  |   +1.000000 |

## Very High Positive Correlation (0.95 ≤ r < 1.00)

| Feature 1                   | Feature 2                   | Correlation |
|-----------------------------|-----------------------------|------------:|
| Bwd Avg Bytes/Bulk          | Bwd Avg Packets/Bulk        |   +0.999961 |
| Packet Length Std           | Payload Std                 |   +0.999955 |
| Flow IAT Max                | Idle Max                    |   +0.999663 |
| TTL Bwd Mean                | TTL Bwd Max                 |   +0.999367 |
| Packet Length Mean          | Payload Mean                |   +0.999224 |
| Average Packet Size         | Payload Mean                |   +0.999224 |
| Total Bwd Packets           | Bwd Header Length           |   +0.999139 |
| Subflow Bwd Packets         | Subflow Bwd Bytes           |   +0.998374 |
| TTL Bwd Mean                | TTL Bwd Min                 |   +0.998197 |
| TTL Fwd Mean                | TTL Fwd Max                 |   +0.998004 |
| Total Bwd Packets           | Total Length of Bwd Packets |   +0.997506 |
| TTL Bwd Min                 | TTL Bwd Max                 |   +0.997122 |
| Total Length of Bwd Packets | Bwd Header Length           |   +0.996973 |
| Flow Duration               | Bwd IAT Total               |   +0.994606 |
| TTL Fwd Mean                | TTL Fwd Min                 |   +0.994287 |
| Flow IAT Max                | Bwd IAT Max                 |   +0.993245 |
| Bwd IAT Max                 | Idle Max                    |   +0.992894 |
| Idle Mean                   | Idle Max                    |   +0.992844 |
| Idle Mean                   | Idle Min                    |   +0.992756 |
| Flow IAT Max                | Idle Mean                   |   +0.992575 |
| Total Fwd Packets           | Fwd Header Length           |   +0.992569 |
| Total Fwd Packets           | Fwd Header Length.1         |   +0.992569 |
| Fwd Packet Length Min       | Packet Length Min           |   +0.991795 |
| Bwd Packet Length Max       | Packet Length Max           |   +0.989711 |
| Bwd Packet Length Max       | Payload Max                 |   +0.989711 |
| Bwd Header Length           | ACK Flag Count              |   +0.987282 |
| TTL Fwd Min                 | TTL Fwd Max                 |   +0.986650 |
| Bwd IAT Max                 | Idle Mean                   |   +0.986378 |
| Total Bwd Packets           | ACK Flag Count              |   +0.985362 |
| Total Length of Bwd Packets | ACK Flag Count              |   +0.985328 |
| Active Mean                 | Active Min                  |   +0.983607 |
| Bwd PSH Flags               | PSH Flag Count              |   +0.982763 |
| Init_Win_bytes_backward     | TCP Window Bwd Max          |   +0.979962 |
| Bwd Packet Length Max       | Bwd Packet Length Std       |   +0.977167 |
| Fwd Packet Length Max       | Fwd Packet Length Std       |   +0.972936 |
| Flow Packets/s              | Fwd Packets/s               |   +0.972565 |
| Packet Length Min           | Payload Min                 |   +0.971846 |
| Idle Max                    | Idle Min                    |   +0.971701 |
| Flow IAT Max                | Idle Min                    |   +0.971533 |
| Bwd Packet Length Std       | Packet Length Max           |   +0.968180 |
| Bwd Packet Length Std       | Payload Max                 |   +0.968180 |
| Bwd IAT Max                 | Idle Min                    |   +0.965718 |
| Fwd Packet Length Min       | Payload Min                 |   +0.963881 |
| Bwd Packet Length Std       | Payload Std                 |   +0.961386 |
| Bwd Packet Length Std       | Packet Length Std           |   +0.961092 |
| Bwd Packet Length Mean      | Payload Mean                |   +0.959495 |
| Avg Bwd Segment Size        | Payload Mean                |   +0.959495 |
| Average Packet Size         | Avg Bwd Segment Size        |   +0.959233 |
| Bwd Packet Length Mean      | Packet Length Mean          |   +0.959233 |
| Packet Length Mean          | Avg Bwd Segment Size        |   +0.959233 |
| Bwd Packet Length Mean      | Average Packet Size         |   +0.959233 |
| Bwd IAT Std                 | Bwd IAT Max                 |   +0.955535 |
| Packet Length Max           | Payload Std                 |   +0.953548 |
| Payload Std                 | Payload Max                 |   +0.953548 |
| Packet Length Max           | Packet Length Std           |   +0.953493 |
| Packet Length Std           | Payload Max                 |   +0.953493 |
| Bwd Packet Length Max       | Payload Std                 |   +0.950696 |
| Bwd Packet Length Max       | Packet Length Std           |   +0.950385 |

## High Positive Correlation (0.90 ≤ r < 0.95)

| Feature 1               | Feature 2              | Correlation |
|-------------------------|------------------------|------------:|
| Flow IAT Max            | Bwd IAT Std            |   +0.949054 |
| Bwd IAT Std             | Idle Max               |   +0.948739 |
| Active Mean             | Active Max             |   +0.948566 |
| Flow IAT Std            | Flow IAT Max           |   +0.937994 |
| Flow IAT Std            | Idle Max               |   +0.937912 |
| Fwd IAT Std             | Fwd IAT Max            |   +0.936517 |
| Bwd IAT Std             | Idle Mean              |   +0.934913 |
| Packet Length Std       | Packet Length Variance |   +0.932499 |
| Packet Length Variance  | Payload Std            |   +0.932209 |
| Bwd Packet Length Mean  | Packet Length Variance |   +0.929232 |
| Packet Length Variance  | Avg Bwd Segment Size   |   +0.929232 |
| Flow IAT Std            | Bwd IAT Max            |   +0.926943 |
| Flow IAT Std            | Idle Mean              |   +0.925397 |
| TCP Window Fwd Mean     | TCP Window Fwd Max     |   +0.920083 |
| TCP Window Bwd Mean     | TCP Window Bwd Max     |   +0.916988 |
| Init_Win_bytes_backward | TCP Window Bwd Mean    |   +0.911709 |
| Bwd IAT Std             | Idle Min               |   +0.906925 |
| dst_port_unique_cnt     | dst_port_entropy       |   +0.903267 |

## High Negative Correlation

| Feature 1               | Feature 2               | Correlation |
|-------------------------|-------------------------|------------:|
| Protocol                | min_seg_size_forward    |   -0.897108 |
| Bwd Packet Length Min   | min_seg_size_forward    |   -0.746164 |
| dst_port_entropy        | TCP Window Bwd Mean     |   -0.719976 |
| min_seg_size_forward    | TTL Bwd Min             |   -0.678628 |
| min_seg_size_forward    | TTL Bwd Mean            |   -0.677385 |
| min_seg_size_forward    | TTL Bwd Max             |   -0.676748 |
| Packet Length Min       | min_seg_size_forward    |   -0.669473 |
| Fwd Packet Length Min   | min_seg_size_forward    |   -0.659297 |
| min_seg_size_forward    | Payload Min             |   -0.654938 |
| Protocol                | TCP Window Bwd Max      |   -0.654316 |
| dst_port_unique_cnt     | TCP Window Bwd Mean     |   -0.646992 |
| Protocol                | Init_Win_bytes_backward |   -0.642914 |
| Protocol                | Fwd Packet Length Std   |   -0.640832 |
| RST Flag Count          | dst_port_entropy        |   -0.638103 |
| dst_port_entropy        | TCP Window Bwd Max      |   -0.630593 |
| Init_Win_bytes_backward | dst_port_entropy        |   -0.625178 |
| Protocol                | Bwd Packet Length Std   |   -0.623697 |
| Protocol                | RST Flag Count          |   -0.607903 |
| Protocol                | Packet Length Max       |   -0.605879 |
| Protocol                | Payload Max             |   -0.605879 |

## Observations

The correlation matrix shows heavy redundancy concentrated in a few families of measurements: packet/byte counts and their subflow equivalents, forward/backward header lengths, packet-length summary statistics (mean/std/variance vs. their "payload" and "segment size" counterparts), inter-arrival-time (IAT) statistics against idle-time statistics, and per-direction TTL and TCP window statistics. Several groups of features are effectively re-derivations of the same underlying flow property, which is expected given how CICIDS-style flow features are engineered (raw counts, per-subflow counts, and rolling summary statistics are often computed from the same base quantities).

## Duplicate Features

| Original Feature       | Duplicate Feature    |
|------------------------|----------------------|
| Fwd Packet Length Mean | Avg Fwd Segment Size |
| Bwd Packet Length Mean | Avg Bwd Segment Size |
| Fwd Header Length      | Fwd Header Length.1  |
| Packet Length Max      | Payload Max          |
| Packet Length Mean     | Average Packet Size  |

These pairs exhibit perfect correlation (r = 1.0) and should be investigated for exact row-wise duplication before feature selection. `Fwd Header Length.1` in particular looks like a column that was accidentally duplicated during feature export (identical name, decimal suffix), and is a strong candidate for immediate removal rather than further investigation.

## Equivalent or Highly Related Statistics

| Feature                 | Equivalent / Related Feature                                                                       |
|-------------------------|----------------------------------------------------------------------------------------------------|
| Packet Length Mean      | Average Packet Size (r = 1.00), Payload Mean (r = 0.999)                                           |
| Bwd Packet Length Mean  | Avg Bwd Segment Size (r = 1.00), Payload Mean (r = 0.96)                                           |
| Packet Length Std       | Payload Std (r = 0.9999), Packet Length Variance (r = 0.93)                                        |
| Packet Length Max       | Payload Max (r = 1.00), Bwd Packet Length Max (r = 0.99)                                           |
| Total Bwd Packets       | Total Length of Bwd Packets (r = 0.998), Bwd Header Length (r = 0.999), ACK Flag Count (r = 0.985) |
| TTL Bwd Mean            | TTL Bwd Min (r = 0.998) / TTL Bwd Max (r = 0.999)                                                  |
| TTL Fwd Mean            | TTL Fwd Min (r = 0.994) / TTL Fwd Max (r = 0.998)                                                  |
| Flow IAT Max            | Idle Max (r = 0.9997), Idle Mean (r = 0.993), Bwd IAT Max (r = 0.993)                              |
| Idle Mean               | Idle Min (r = 0.993) / Idle Max (r = 0.993)                                                        |
| Init_Win_bytes_backward | TCP Window Bwd Max (r = 0.98), TCP Window Bwd Mean (r = 0.91)                                      |
| dst_port_unique_cnt     | dst_port_entropy (r = 0.90)                                                                        |

These features describe closely related statistical properties (the same underlying metric measured as a mean, min, max, or an alias field like "Payload" vs. "Packet Length" vs. "Segment Size") and likely provide overlapping information for modeling purposes.

## Suspicious Correlations

| Feature 1         | Feature 2            | Correlation |
|-------------------|----------------------|------------:|
| Bwd Header Length | ACK Flag Count       |   +0.987282 |
| Total Bwd Packets | ACK Flag Count       |   +0.985362 |
| Protocol          | min_seg_size_forward |   -0.897108 |
| Protocol          | TCP Window Bwd Max   |   -0.654316 |
| RST Flag Count    | dst_port_entropy     |   -0.638103 |
